In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
from google.colab import files
uploaded=files.upload

In [3]:
spark = (SparkSession.builder.appName("SmartHomeEnergyTracker").getOrCreate())

In [6]:
df = spark.read.csv(
    "energy_usage.csv",
    header=True,
    inferSchema=True
)

In [12]:
df = df.withColumn(
    "timestamp",
    to_timestamp("timestamp")
)

In [17]:
df = df.withColumn(
    "usage_type",
    when(
        (hour("timestamp") >= 6)
        &
        (hour("timestamp") < 22),
        "Peak"
    )
    .otherwise("Off-Peak")
)
df.show()

+---------+-------+-------------------+----------+----------+
|device_id|room_id|          timestamp|energy_kwh|usage_type|
+---------+-------+-------------------+----------+----------+
|        1|      1|2026-06-01 07:30:00|       2.5|      Peak|
|        2|      2|2026-06-01 09:15:00|       5.0|      Peak|
|        3|      3|2026-06-01 13:45:00|       3.2|      Peak|
|        4|      1|2026-06-01 18:20:00|       1.8|      Peak|
|        5|      2|2026-06-01 22:30:00|       4.1|  Off-Peak|
|        1|      1|2026-06-02 08:00:00|       2.8|      Peak|
|        2|      2|2026-06-02 12:15:00|       4.7|      Peak|
|        3|      3|2026-06-02 15:40:00|       3.5|      Peak|
|        4|      1|2026-06-02 19:10:00|       2.0|      Peak|
|        5|      2|2026-06-02 23:00:00|       4.5|  Off-Peak|
|        1|      1|2026-06-03 07:45:00|       2.6|      Peak|
|        2|      2|2026-06-03 10:20:00|       5.3|      Peak|
|        3|      3|2026-06-03 14:10:00|       3.8|      Peak|
|       

In [14]:
device_usage = (
    df.groupBy(
        "device_id",
        "usage_type"
    )
    .agg(
        sum("energy_kwh")
        .alias("total_energy")
    )
)
device_usage.show()

+---------+----------+------------------+
|device_id|usage_type|      total_energy|
+---------+----------+------------------+
|        4|      Peak|               8.1|
|        1|      Peak|              10.8|
|        2|      Peak|              20.1|
|        5|  Off-Peak|13.399999999999999|
|        5|      Peak|               4.3|
|        3|      Peak|              14.1|
+---------+----------+------------------+



In [15]:
top_devices = (
    df.groupBy("device_id")
    .agg(
        sum("energy_kwh")
        .alias("total_energy")
    )
    .orderBy(
        col("total_energy")
        .desc()
    )
)
top_devices.show()

+---------+------------+
|device_id|total_energy|
+---------+------------+
|        2|        20.1|
|        5|        17.7|
|        3|        14.1|
|        1|        10.8|
|        4|         8.1|
+---------+------------+



In [16]:
top_devices.write \
.mode("overwrite") \
.option("header", True) \
.csv("top_energy_devices.csv")